<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Adamastor_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Adamastor-GPT**

Este modelo é baseado na arquitetura <b> Transformer </b>, inspirado nos modelos do tipo <b> GPT (<i>Generative Pre-trained Transformer</i>)</b>. <br>

Ao longo deste Colab, iremos navegar pelas componentes cruciais dos modelos GPT-like, que atualmente dominam grande parte do mundo da Inteligência Artificial.

<b> No final, teremos um modelo estilo GPT funcional, capaz de realizar inferência de forma limpa e consistente. </b>

O treino vai ser realizado com uma <b> T4 GPU</b>.


### **Libraries**

In [12]:
from tokenizers import Tokenizer #Tokenization
import torch
import torch.nn as nn
import torch.nn.functional as F
#import math

device = "cuda" if torch.cuda.is_available() else "cpu" #Modelos LLM rodam em GPU e não em CPU
print (device)

cuda


### **Dados do Modelo**

In [13]:
Sequence_Length = 32
Batch_Size = 16
Embedding_Dimension = 64
Head_Size = 8 #Head_Size = Embedding_Dimension // Number_Head
Number_Head = 8
Blocos = 4

###


---

### **Tokenization**


In [15]:
Tokenizer = Tokenizer.from_file ("GAMA") #Tokenizer já criado posteriormente baseado no algoritmo BPE https://github.com/Eliezer-Carvalho/Adamastor-GPT/tree/main/data

with open ("Os Lusiadas.txt", "r", encoding = "utf-8") as f: #Dataset pode ser encontrado rep https://github.com/Eliezer-Carvalho/Adamastor-GPT/tree/main/data
  dataset = f.read()

Tokenization = Tokenizer.encode (dataset) #Tokenization dos dados
print (len(Tokenization.ids))
print (len(Tokenization.tokens))

tensor = torch.tensor (Tokenization.ids, dtype = torch.long) #Tensor Shape
print (tensor [:50]) #50 primeiros Tokens

teste = Tokenizer.encode ("Os Lusiadas são fixes")
print (teste.tokens)
print (teste.ids)

n = int (0.9 * (len(tensor)))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

74830
74830
tensor([  26, 4359,   23,   32,   30,   69,   13,   16,   13, 4359,    0,   23,
        1526,   94,  679,  119,  527,    0, 2146,   21,    2,    0,  342,  670,
        1060, 2561,  527, 2459,  396,  142,  137, 2852,  114,  206, 4180,  781,
        3292, 2513,  573, 4572,  692,  396, 1997,  660,   38,  541,  113,  362,
         137,   31])
['Os ', 'Lus', 'i', 'adas ', 'são ', 'fi', 'x', 'es']
[280, 427, 46, 788, 610, 174, 59, 103]


###


---

### **Batch e Sequence Length**

In [17]:
torch.manual_seed (2000)

def Batch (split):

  dados = dados_treino if split == 'treino' else dados_teste #Posteriormente no treino vai ser importante para testar o modelo com dados de treino e com dados de teste

  idx = torch.randint (0, len(dados_treino) - Sequence_Length, (Batch_Size, ))
  x = torch.stack ([dados_treino [i: i + Sequence_Length] for i in idx])
  y = torch.stack ([dados_treino [i + 1: i + Sequence_Length + 1] for i in idx])

  x, y = x.to(device), y.to(device) #Passar os dados para a GPU

  return x, y

x, y = Batch('train')

print (f"(B, T) = {x.shape}") #(B, T)
print (x[1])

print (f"(B, T) = {y.shape}") #(B, T)
print (y[1])

(B, T) = torch.Size([16, 32])
tensor([1061,   89,  150,  108,   27, 4417, 1116, 4659, 2880,  105,  727, 4563,
         144,  395,  115,  147, 1254, 2778,  272,  353,  472, 1611, 2883,  224,
         330, 1334,    0, 2560, 1186,  604,  343,  728], device='cuda:0')
(B, T) = torch.Size([16, 32])
tensor([  89,  150,  108,   27, 4417, 1116, 4659, 2880,  105,  727, 4563,  144,
         395,  115,  147, 1254, 2778,  272,  353,  472, 1611, 2883,  224,  330,
        1334,    0, 2560, 1186,  604,  343,  728,  763], device='cuda:0')


###


---



### **Definição de uma Head**

Uma Head não é nada mais nada menos do que a aplicação do mecanismo <b> Attention</b>.

$$
{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

<p align = "center">
  <img src = "https://machinelearningmastery.com/wp-content/uploads/2021/09/tour_3.png" width = 250>
</p>



A <b> Head Size (tamanho da cabeça)</b> representa o profundidade que uma Single Head irá ter. <br>
Ou seja, cada head não “vê” a sequência inteira do embedding diretamente. <br>
Transforma os embeddings para um espaço mais pequeno (<b>head_size</b>), onde aprende relações diferentes. <br>

Desta maneira cada head aprende padrões diferentes (ex: sintático, semântico, posição, etc.)

$$
head{size} = \frac{d_{emb}}{heads}
$$

In [18]:
#Uso de POO, standard na indústria e mais prático

class Head (nn.Module): #Head = Self Masked Attention
  def __init__(self):
    super().__init__()
    self.Query = nn.Linear (Embedding_Dimension, Head_Size, bias = False) #Isto é praticamente uma MLP sem bias
    self.Key = nn.Linear (Embedding_Dimension, Head_Size, bias = False) #Isto é praticamente uma MLP sem bias
    self.Value = nn.Linear (Embedding_Dimension, Head_Size, bias = False) #Isto é praticamente uma MLP sem bias

  def forward (self, x):
    B, T, C = x.shape

    Q = self.Query (x)
    K = self.Key (x)
    V = self.Value (x)

    # Scaled Dot-Product Attention
    attention_score = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5)

    tril = torch.tril (torch.ones (T, T, device = x.device))
    attention_score = attention_score.masked_fill (tril == 0, float ("-inf")) #Self Masked Attention

    attention_score = F.softmax (attention_score, dim = -1) #Normalização da attention_score

    output = attention_score @ V
    return output

###


---


### **Definição de Multi Head** ⭐

É um dos conceitos mais simples e importantes dentro da arquitetura GPT.

Consiste em concatenar os outputs de várias <b> <i> heads (cabeças de atenção) </i> </b> e, em seguida, aplicar uma <i> <b> output projection </b> </i> para combinar e misturar a informação obtida.

As <b> Multi-Heads </b> permitem que o modelo capte diferentes tipos de relações dentro de uma sequência <b> (Sequence Length)</b>, analisando-a sob múltiplas perspetivas em simultâneo.

$$
{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O
$$

$$
{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

<p align = "center">
  <img src = "https://www.jeremyjordan.me/content/images/size/w1000/2023/05/multi-head-attention.png" width = "600">
</p>

In [20]:
class Multi_Head (nn.Module):
  def __init__(self):
    super().__init__()
    self.Concat = nn.ModuleList ([Head() for i in range (Number_Head)]) #Head Concatenation #Definição de Multi Head Attention #Concat o output do Número de Heads
    self.Output_Projection = nn.Linear (Embedding_Dimension, Embedding_Dimension) #Necessário para misturar a informação das Heads #Desta forma volta também à dimensão (B, T, C)

  def forward (self, x):

    output = self.Output_Projection (torch.cat ([h (x) for h in self.Concat], dim = -1))
    return output


###


---


### **Definição de um Block ([Pre-LayerNorm Transformer](https://medium.com/@ashutoshs81127/why-pre-norm-became-the-default-in-transformers-4229047e2620))** ⭐

Com o paper <i> “Attention Is All You Need”</i>, foi introduzida a arquitetura Transformer, originalmente com uma abordagem chamada <b> Post-Norm</b>. <br>
No entanto, rapidamente os investigadores identificaram problemas ao escalar modelos mais profundos, como:

- <b> Treino instável em redes profundas </b>
- <b> Gradientes a explodir ou a desaparecer </b>

Isto levou ao surgimento da abordagem <b> Pre-Norm</b>, que melhorou significativamente a estabilidade do treino.
<br>
<br>
#### <b> Post-Norm vs Pre-Norm ? </b>

- <b>Abordagem Post-Norm:</b>

<i> Attention -> Add -> Norm -> Feed Forward -> Add -> Norm</i>

- <b>Abordagem Pre-Norm:</b>

<i> Norm -> Attention -> Add -> Norm -> Feed Forward -> Add </i>

<br>

Nos modelos atuais, o bloco típico é <b> Pre-Norm</b>, e a ordem correta é:
<b>
- Layer Norm
- Multi-Head Self-Attention
- Residual Connection (Add)
- Layer Norm
- Feed Forward Neural Network (MLP)
- Residual Connection (Add)
</b>

Estes métodos conjugados é aquilo a que chamamos de <b> Block </b> ou <b> Layer</b>.
<br>

<p align = "center">
  <img src = "https://miro.medium.com/v2/resize:fit:1358/1*PGF3P2R9xaP7sB_NwmlNZw.png" width = 625>
</p>

In [ ]:
class Block (nn.Module):
  def __init__(self):
    super().__init__()

    self.LayerNorm_Attention = nn.LayerNorm (Embedding_Dimension) #Layer Norm
    self.Masked_Multi_Head_Attention = Multi_Head() #Self Masked Attention


    self.LayerNorm_FNN = nn.LayerNorm (Embedding_Dimension) #Layer Norm
    self.FeedForward_NeuralNet = nn.Sequential ( #MLP
        nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
        nn.ReLU(),
        nn.Linear (Embedding_Dimension * 4, Embedding_Dimension)
    )


  def forward (self, x):
    Norm_Attention = self.Masked_Multi_Head_Attention (self.LayerNorm_Attention (x)) #Norm #LayerNorm
    Add_Attention = x + Norm_Attention #Add #Residual Connection
    x = Add_Attention

    Norm_FNN = self.FeedForward_NeuralNet (self.LayerNorm_FNN (x)) #Norm #LayerNorm
    Add_FNN = x + Norm_FNN #Add #Residual Connection
    x = Add_FNN

    return x

###


---


### **Modelo Adamastor-GPT**

<i> Implementação do Modelo Final </i>

In [ ]:
class ADAMASTOR_GPT (nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding = nn.Embedding (Tokenizer.get_vocab_size(), Embedding_Dimension) #Embedding dos Tokens
    self.Positional_Encoding = nn.Embedding (Sequence_Length, Embedding_Dimension) #Positional Encoding dos Tokens

    self.Blocks = nn.Sequential ( #Chamada do número de blocos ou layers
        Block(),
        Block(),
        Block(),
        Block()
    )

    self.Language_Modeling = nn.Linear (Embedding_Dimension, Tokenizer.get_vocab_size()) #Converte o vetor final do modelo em logits sobre o vocabulário inteiro.

  def forward (self, x):
    B, T = x.shape

    x = self.Embedding (x) + self.Positional_Encoding (torch.arange (T, device = x.device))

    x = self.Blocks(x)

    logits = self.Language_Modeling (x)

    return logits #Return de um tensor (4, 8, 5000)

###


---


### **Treino do Modelo (Super-Importante)**

Como já reparámos neste código que estamos a construir, os pesos são valores totalmente aleatórios por isso é necessário haver uma maneira de treinar estes parâmetros


#### Função que Treina o modelo

In [ ]:
def training (modelo, steps):

  modelo.train() #Ativa o treino

  for i in range (steps):

    x, y = Batch('train')

    logits = modelo (x)

    #print (logits.shape) #(16,32,5000)
    B, T, V = logits.shape #(Batch, Sequence Length, Vocab_Size)

    loss = loss_function (
        logits.view (B * T, V),
        y.view (B * T)
    ) #Entrada que o Cross Entropy espera

    optimizer.zero_grad() #Zera os gradientes acumulados nos parâmetros.
    loss.backward() #Backpropagation

    optimizer.step() #Optimizer aplicado

    if i % 1000 == 0: #Printa de 1000 em 1000

      losses = loss_calc (modelo)
      print(f"Step {i} | Train Loss = {losses['train']:.4f} | Validation Loss : {losses['val']:.4f}")


#### Calcular a Loss

In [ ]:
#https://github.com/karpathy/nanoGPT/blob/master/train.py
@torch.no_grad()
def loss_calc (modelo, loop_eval = 500):

    output = {}
    modelo.eval() #Desliga modo de treino

    for split in ['train', 'val']: #Para calcular a loss para os dois conjuntos de dados

        losses = torch.zeros (loop_eval) #Cria um tensor com int "0" com tamanho loop_eval

        for k in range (loop_eval): #Por cada loop_eval o modelo vai ser alimentado por um novo batch de dados e vai ser testado a sua habilidade tanto em dados de teste como em dados de validação

            x, y = Batch (split) #Quando for train:                         #Quando for val:
            logits = modelo (x) #Alimenta o modelo com dados de treino      #Alimenta o modelo com dados de teste

            B, T, V = logits.shape

            loss = loss_function (
                logits.view (B * T, V),
                y.view (B * T)
            )

            losses[k] = loss.item() #Cria uma lista com as losses todas

        output[split] = losses.mean() #Calcula a média, mais preciso

    modelo.train() #Volto ao modo de treino
    return output

### Adamastor-GPT Treino

In [ ]:
modelo = ADAMASTOR_GPT().to(device)
#print (modelo)
optimizer = torch.optim.AdamW (modelo.parameters(), lr = 3e-4) #Usa-se AdamW ou SGD #Responsável por atualizar os pesos do modelo
#print (optimizer)
loss_function = nn.CrossEntropyLoss() #Função de perda que mede quão diferente está a previsão do modelo da resposta correta.

training (modelo, 50000)

Step 0 | Train Loss = 8.8879 | Validation Loss : 8.8877
Step 1000 | Train Loss = 7.7862 | Validation Loss : 7.7880
Step 2000 | Train Loss = 7.7002 | Validation Loss : 7.6911
Step 3000 | Train Loss = 7.5889 | Validation Loss : 7.5901
Step 4000 | Train Loss = 7.4127 | Validation Loss : 7.4080
Step 5000 | Train Loss = 7.1764 | Validation Loss : 7.1731
Step 6000 | Train Loss = 6.9026 | Validation Loss : 6.9047
Step 7000 | Train Loss = 6.6198 | Validation Loss : 6.6262
Step 8000 | Train Loss = 6.3694 | Validation Loss : 6.3844
Step 9000 | Train Loss = 6.1673 | Validation Loss : 6.1588
Step 10000 | Train Loss = 5.9659 | Validation Loss : 5.9727
Step 11000 | Train Loss = 5.8004 | Validation Loss : 5.7926
Step 12000 | Train Loss = 5.6421 | Validation Loss : 5.6483
Step 13000 | Train Loss = 5.5013 | Validation Loss : 5.5039
Step 14000 | Train Loss = 5.3687 | Validation Loss : 5.3683
Step 15000 | Train Loss = 5.2526 | Validation Loss : 5.2421
Step 16000 | Train Loss = 5.1288 | Validation Loss : 

###


---

### Geração de Texto

In [ ]:
@torch.no_grad()
def gerar_texto (modelo, max_tokens, indice, temperatura = 1, top_tokens = None):

  modelo.eval()

  for i in range (max_tokens): #Vai adicionando token a token

    indice_condition = indice [:, -Sequence_Length:] #Alimento o modelo com o máximo de tokens possíveis, não posso dar um contexto maior do que daquele com que treinou

    logits = modelo (indice_condition) #Aplicação da Arquitetura

    logits = logits [:, -1, :] / temperatura #Foca no que gerou anteriormente #Temperatura aumenta ou diminui a creatividade. Se > 1 há mais chance de tokens raros serem escolhidos

    #https://github.com/karpathy/nanoGPT/blob/master/model.py
    if top_tokens is not None: #TopK funciona como um limitador que ajuda o modelo a olhar para os tokens com  melhores probabilidades, em vez de olhar para todos
              V = torch.topk (logits, min(top_tokens, logits.size (-1))) #https://docs.pytorch.org/docs/2.11/generated/torch.topk.html
              logits [logits < V [:, [-1]]] = -float('Inf') #Substitui os tokens com menor probabilidade por -inf


    probs = F.softmax (logits, dim = -1) #Aplica Softmax para ter as probabilidades

    prev = torch.multinomial (probs, num_samples = 1) #Seleciona um token

    indice = torch.cat ((indice, prev), dim  = 1) #Adiciona esse token ao contexto

  return indice

### Adamastor-GPT Inferência

In [ ]:
contexto = torch.zeros ((1, 1), dtype = torch.long, device = device) #Cria um tensor [1,1] com o valor 0 e à medida que entra na função vai adicionando tokens
#print (contexto)
indice = gerar_texto (modelo, max_tokens = 200, indice = contexto)

#print (indice)
#print (indice.shape)

print (Tokenizer.decode(indice [0,1:].tolist()))

 Vendo  a  Quer ia  e a  antig da  fam éli cos e  s ei,  enfim, que  lho  tinha  o fals o pens ament e. « Bem  pera  trás  fi cav esta  a  razão  lhe  der  dos  sete 


###


---

### Características do Modelo

In [ ]:
'''
print (modelo.state_dict())
for param in modelo.state_dict():
  print (param)
'''

'\nprint (modelo.state_dict())\nfor param in modelo.state_dict():\n  print (param)\n'

#### Número de Parâmetros do Modelo

In [ ]:
print (sum(p.numel() for p in modelo.parameters()))
#save = torch.save(modelo.state_dict(), "Adamastor-GPT.pth")

846216


###


---

## Notas

O teu ADAMASTOR_GPT é:

um Transformer decoder-only (tipo GPT)
com:
4 camadas (blocos)
8 cabeças de atenção
embeddings de tamanho 64
vocabulário de 5000 tokens
contexto máximo de 32 tokens

FAZER ESTA LEITURA

interpretar loss value, o que o valor quer dizer ? <br>

CPU do PC vs TPU - tempo a treinar o modelo e resultados obtidos

Dropout é uma técnica de regularização usada para evitar overfitting.

In [ ]:
'''

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster
'''

'\n\n@dataclass\nclass GPTConfig:\n    block_size: int = 1024\n    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency\n    n_layer: int = 12\n    n_head: int = 12\n    n_embd: int = 768\n    dropout: float = 0.0\n    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster\n'